In [4]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [5]:
merged_df = pd.read_parquet("../Model_Data/merged_df_50Hz.parquet")

In [6]:
RANDOM_STATE  = 42
TEST_FRACTION = 0.2
JESSICA       = "Jessica_Schmid"   # alle anderen Authors landen komplett im Test

# Author pro ID (jede Mess-ID gehoert zu genau einem Author)
author_per_id = merged_df.groupby("ID")["Author"].first()

# IDs pro Tag 
ids_per_class = merged_df.groupby("Tag")["ID"].unique().to_dict()

rng = np.random.default_rng(RANDOM_STATE)

train_ids = []
test_ids  = []

for tag, ids in ids_per_class.items():
    ids        = np.array(ids)
    authors    = author_per_id.loc[ids].values
    is_jessica = (authors == JESSICA)

    forced_test_ids = ids[~is_jessica]   # alle Raphael/Silas -> Test
    jessica_ids     = ids[is_jessica]

    # Wieviele Jessica-IDs zusaetzlich in Test, damit dieser Tag ~20% im Test hat
    target_test    = int(round(len(ids) * TEST_FRACTION))
    n_jessica_test = max(0, target_test - len(forced_test_ids))
    n_jessica_test = min(n_jessica_test, len(jessica_ids))

    if n_jessica_test > 0:
        jessica_test = rng.choice(jessica_ids, size=n_jessica_test, replace=False)
    else:
        jessica_test = np.array([], dtype=jessica_ids.dtype)

    tag_test_ids  = np.concatenate([forced_test_ids, jessica_test])
    tag_train_ids = np.setdiff1d(jessica_ids, jessica_test)

    test_ids.extend(tag_test_ids.tolist())
    train_ids.extend(tag_train_ids.tolist())

# Train / Test DataFrames
df_train = merged_df[merged_df["ID"].isin(train_ids)].copy()
df_test  = merged_df[merged_df["ID"].isin(test_ids)].copy()

n_ids = merged_df["ID"].nunique()
print(f"Train shape: {df_train.shape}")
print(f"Test  shape: {df_test.shape}")
print(f"Test fraction (rows): {len(df_test) / len(merged_df):.1%}")
print(f"Test fraction (IDs):  {len(test_ids) / n_ids:.1%}")
print()
print("Authors im Train:", sorted(df_train["Author"].unique()))
print("Authors im Test: ", sorted(df_test["Author"].unique()))
print()
print("Test-Verteilung pro Author:")
print(df_test.groupby("Author")["ID"].nunique().rename("Mess-IDs"))
print()
print("Test-Verteilung pro Tag (IDs):")
print(df_test.groupby("Tag")["ID"].nunique().rename("Test-IDs"))

Train shape: (709250, 17)
Test  shape: (397677, 17)
Test fraction (rows): 35.9%
Test fraction (IDs):  26.4%

Authors im Train: ['Jessica_Schmid']
Authors im Test:  ['Jessica_Schmid', 'Raphael_Schuepbach', 'Silas_Tschopp']

Test-Verteilung pro Author:
Author
Jessica_Schmid        108
Raphael_Schuepbach      9
Silas_Tschopp          80
Name: Mess-IDs, dtype: int64

Test-Verteilung pro Tag (IDs):
Tag
Auto         25
Laufen       25
Lift         32
Roundkick    15
Treppe       20
Velo         57
Zug          23
Name: Test-IDs, dtype: int64


In [7]:
def create_sliding_windows_full(df, feature_cols, label_col="Tag",
                                window_size=100, step_size=50):

    
    X = []
    y = []

    for id_, group in df.groupby("ID"):
        group = group.sort_values("time_elapsed")
        data = group[feature_cols].values
        label = group[label_col].iloc[0]  # eine Bewegung pro Aufnahme
        
        for start in range(0, len(group) - window_size + 1, step_size):
            end = start + window_size
            window = data[start:end]
            
            X.append(window)
            y.append(label)
    
    X = np.array(X)
    y = np.array(y)
    return X, y

In [8]:
feature_cols = [
    "x_acc", "y_acc", "z_acc",
    "x_gyr", "y_gyr", "z_gyr",
    "qx", "qy", "qz", "qw",
    "roll", "pitch", "yaw"
]

WINDOW_SIZE = 100
STEP_SIZE = 50

X_train, y_train = create_sliding_windows_full(df_train, feature_cols,
                                               window_size=WINDOW_SIZE,
                                               step_size=STEP_SIZE)

X_test, y_test = create_sliding_windows_full(df_test, feature_cols,
                                             window_size=WINDOW_SIZE,
                                             step_size=STEP_SIZE)

In [9]:
np.savez_compressed("../Model_data/train.npz", X=X_train, y=y_train)
np.savez_compressed("../Model_data/test.npz", X=X_test, y=y_test)